In [ ]:
import pandas as pd
import os

# ============================================================
# 1. KHAI BÁO ĐƯỜNG DẪN FILE NGUỒN
# ============================================================

product_path = "/content/product_new.csv"


# ============================================================
# 2. KIỂM TRA FILE CÓ TỒN TẠI HAY KHÔNG
# ============================================================

if not os.path.exists(product_path):
    raise FileNotFoundError("Không tìm thấy file product_new.csv trong /content/")


# ============================================================
# 3. ĐỌC DỮ LIỆU TỪ CSV
# ============================================================

product_df = pd.read_csv(product_path)


# ============================================================
# 4. HÀM CHUẨN HÓA TÊN CỘT
# Ví dụ:
# "Product ID"  -> "product_id"
# " PRICE "     -> "price"
# ============================================================

def normalize_columns(df):
    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()                     # Xóa khoảng trắng đầu/cuối
        .str.lower()                     # Chuyển về chữ thường
        .str.replace(" ", "_", regex=False)  # Khoảng trắng -> _
    )

    return df


product_df = normalize_columns(product_df)


# ============================================================
# 5. TẠO DIM_PRODUCT
#
# Star Schema:
# product_key       : Surrogate Key
# product_id        : Business Key từ hệ thống nguồn
# product_name
# category
# segment
# size
# color
# current_price
# current_cogs
# ============================================================

required_product_columns = [
    "product_id",
    "product_name",
    "category",
    "segment",
    "size",
    "color",
    "price",
    "cogs"
]


# Kiểm tra các cột cần thiết có tồn tại trong file nguồn hay không
missing_product_columns = [
    col for col in required_product_columns
    if col not in product_df.columns
]

if missing_product_columns:
    raise ValueError(
        f"product_new.csv đang thiếu các cột: {missing_product_columns}"
    )


# Chỉ lấy những cột cần thiết cho DIM_PRODUCT
dim_product = product_df[required_product_columns].copy()


# ------------------------------------------------------------
# Làm sạch PRODUCT
# ------------------------------------------------------------

# Không thể tạo Dimension cho sản phẩm không có product_id
dim_product = dim_product.dropna(subset=["product_id"])


# Các cột dạng text
product_text_columns = [
    "product_id",
    "product_name",
    "category",
    "segment",
    "size",
    "color"
]

# Xóa khoảng trắng đầu/cuối nhưng vẫn giữ giá trị thiếu là NA
for col in product_text_columns:
    dim_product[col] = (
        dim_product[col]
        .astype("string")
        .str.strip()
    )


# Chuyển price và cogs sang dạng số
# Giá trị không chuyển được sẽ thành NaN
dim_product["price"] = pd.to_numeric(
    dim_product["price"],
    errors="coerce"
)

dim_product["cogs"] = pd.to_numeric(
    dim_product["cogs"],
    errors="coerce"
)


# Nếu một product_id xuất hiện nhiều lần,
# chỉ giữ bản ghi cuối cùng
dim_product = (
    dim_product
    .drop_duplicates(subset=["product_id"], keep="last")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Tạo Surrogate Key cho DIM_PRODUCT
#
# product_key:
# 1
# 2
# 3
# ...
# ------------------------------------------------------------

dim_product.insert(
    0,
    "product_key",
    range(1, len(dim_product) + 1)
)


# Đổi tên price và cogs theo đúng Star Schema
dim_product = dim_product.rename(
    columns={
        "price": "current_price",
        "cogs": "current_cogs"
    }
)


# Sắp xếp lại thứ tự cột đúng theo DIM_PRODUCT
dim_product = dim_product[
    [
        "product_key",
        "product_id",
        "product_name",
        "category",
        "segment",
        "size",
        "color",
        "current_price",
        "current_cogs"
    ]
]


# ============================================================
# 6. XUẤT FILE DIM_PRODUCT
# ============================================================

dim_product.to_csv(
    "/content/dim_product.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 7. THÔNG BÁO HOÀN THÀNH
# ============================================================

print("ETL hoàn tất!")

print("\nĐã tạo:")
print("/content/dim_product.csv")